In [1]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [ ]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [ ]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 3,
        "step_number": 10,
        "position_z": 0,
        "step_size_z": 1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        # chip_name="V1_R_W_2_Right",
        chip_name="SiN_beam",
        sample_name="w64_sweep",  # test_1_right, w=20
        # sample_name="w10_2_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 10.0%
process completed: 20.0%
process completed: 30.0%
process completed: 40.0%
process completed: 50.0%
process completed: 60.0%
process completed: 70.0%
process completed: 80.0%
process completed: 90.0%
process completed: 100.0%
Config saved to: ./result/SiN_beam/202606111453_w64\202606111453_SiN_beam_w64_config.json
Exception occurred: list indices must be integers or slices, not NoneType
Stage Moving Done
Stage disconnected


({'position': [0.0,
   0.990630817590869,
   1.98126163518174,
   2.96884060182501,
   3.9539780877102,
   4.93545335245827,
   5.93340861232337,
   6.93258461256752,
   7.92565691091647,
   8.92544328135014],
  'voltage': [[0.1405949,
    0.1402972,
    0.1405427,
    0.1402263,
    0.1410421,
    0.1395534,
    0.1404054,
    0.1415384,
    0.1411854,
    0.1411312,
    0.142043,
    0.1414595,
    0.1411385,
    0.1402151,
    0.1406459,
    0.1410998],
   [0.1411355,
    0.1417352,
    0.1434378,
    0.1407232,
    0.1422567,
    0.1417611,
    0.1412585,
    0.1418499,
    0.1417895,
    0.1420976,
    0.1426684,
    0.142943,
    0.1421859,
    0.1420522,
    0.1432599,
    0.1427242],
   [0.145845,
    0.1457032,
    0.1448139,
    0.1467478,
    0.1454223,
    0.1446066,
    0.1456108,
    0.1453279,
    0.1463214,
    0.1442991,
    0.1454266,
    0.1473406,
    0.1458258,
    0.1453944,
    0.1465363,
    0.1464181],
   [0.04152675,
    0.04275438,
    0.04136148,
    0.04185

<Figure size 640x480 with 0 Axes>